In [2]:
import numpy as np, pandas as pd, json, joblib, shutil, os, sys, sklearn

USE_FULL = True    # False = 20k (quick test) | True = 100k (final numbers)
path = ("/content/finsight_loan_applications_100k.csv" if USE_FULL
        else "/content/finsight_loan_applications_20k_sample.csv")

df = pd.read_csv(path)
print(path, df.shape)
print("versions:", sys.version.split()[0], pd.__version__, np.__version__, sklearn.__version__)

/content/finsight_loan_applications_100k.csv (13208, 30)
versions: 3.13.18 2.2.3 2.4.6 1.6.1


In [3]:
print("approval rate:", round(df["approved"].mean(), 4))
print(df["split"].value_counts(normalize=True).round(3))
print(df.isnull().mean()[df.isnull().mean() > 0].round(3))

approval rate: 0.4451
split
train    0.700
val      0.151
test     0.148
Name: proportion, dtype: float64
employment_years             0.009
residence_years              0.010
cibil_score                  0.130
credit_age_years             0.130
credit_utilisation_pct       0.139
loan_tenure_months           0.000
assumed_interest_rate_pct    0.000
proposed_emi                 0.000
foir_current                 0.000
foir_post_loan               0.000
loan_to_monthly_income       0.000
documents_complete           0.000
approved                     0.000
primary_rejection_reason     0.445
ever_30dpd_6m                0.555
dtype: float64


In [ ]:
%%writefile features.py
import numpy as np
import pandas as pd

NUMERIC = ["monthly_income", "employment_years", "residence_years", "is_new_to_credit",
           "cibil_score", "credit_age_years", "credit_utilisation_pct",
           "recent_enquiries_6m", "missed_payments_12m", "existing_loans_count",
           "existing_emi_monthly", "loan_amount_requested", "loan_tenure_months",
           "foir_current", "foir_post_loan", "loan_to_monthly_income", "documents_complete"]
CATEGORICAL = ["employment_type"]
FEATURES = NUMERIC + CATEGORICAL
SENSITIVE = ["gender", "city_tier", "age"]


def get_rate(score, is_ntc):
    rate = np.select([score <= 670, score <= 730, score <= 800, score <= 850],
                     [19, 15.5, 13, 11.5], default=10.75)
    return np.where(is_ntc == 1, 17, rate)


def get_emi(amount, rate, months):
    r = rate / 1200
    return amount * r * (1 + r) ** months / ((1 + r) ** months - 1)


def make_features(df):
    d = df.copy()
    rate = get_rate(d["cibil_score"].fillna(0), d["is_new_to_credit"])
    emi = get_emi(d["loan_amount_requested"], rate, d["loan_tenure_months"])
    d["foir_current"] = d["existing_emi_monthly"] / d["monthly_income"]
    d["foir_post_loan"] = (d["existing_emi_monthly"] + emi) / d["monthly_income"]
    d["loan_to_monthly_income"] = d["loan_amount_requested"] / d["monthly_income"]
    return d[FEATURES]

Overwriting features.py


In [ ]:
from features import make_features, FEATURES, NUMERIC, CATEGORICAL, SENSITIVE

X = make_features(df)
y = df["approved"]

X_train, y_train = X[df["split"] == "train"], y[df["split"] == "train"]
X_val,   y_val   = X[df["split"] == "val"],   y[df["split"] == "val"]
X_test,  y_test  = X[df["split"] == "test"],  y[df["split"] == "test"]
print(len(X_train), len(X_val), len(X_test))

69999 15000 15001


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

prep_scaled = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
prep_plain = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])

models = {
    "logistic": Pipeline([("prep", prep_scaled), ("model", LogisticRegression(max_iter=1000))]),
    "tree":     Pipeline([("prep", prep_plain), ("model", DecisionTreeClassifier(max_depth=4, min_samples_leaf=100, random_state=42))]),
    "boosting": Pipeline([("prep", prep_plain), ("model", HistGradientBoostingClassifier(random_state=42))]),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    tr = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
    va = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
    print(f"{name:10s} train {tr:.3f} | val {va:.3f}")

logistic   train 0.875 | val 0.878
tree       train 0.853 | val 0.857
boosting   train 0.901 | val 0.891


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

grids = {
    "logistic": {"model__C": loguniform(1e-3, 1e2)},
    "tree": {"model__max_depth": [3, 4, 5], "model__min_samples_leaf": [50, 100, 200]},
    "boosting": {"model__max_depth": [3, 4, 6, None], "model__learning_rate": [0.03, 0.05, 0.1],
                 "model__max_iter": [200, 400], "model__l2_regularization": [0, 1, 5],
                 "model__min_samples_leaf": [20, 50, 100], "model__max_leaf_nodes": [15, 31]},
}
n_iters = {"logistic": 10, "tree": 9, "boosting": 15}

best_params = {}
for name in list(models):
    s = RandomizedSearchCV(models[name], grids[name], n_iter=n_iters[name], cv=3,
                           scoring="roc_auc", random_state=42, n_jobs=-1)
    s.fit(X_train, y_train)
    models[name] = s.best_estimator_
    best_params[name] = {k: str(v) for k, v in s.best_params_.items()}
    va = roc_auc_score(y_val, models[name].predict_proba(X_val)[:, 1])
    print(f"{name:10s} cv {s.best_score_:.3f} | val {va:.3f} | {best_params[name]}")

logistic   cv 0.874 | val 0.878 | {'model__C': '0.0745934328572655'}
tree       cv 0.862 | val 0.865 | {'model__min_samples_leaf': '50', 'model__max_depth': '5'}
boosting   cv 0.889 | val 0.892 | {'model__min_samples_leaf': '50', 'model__max_leaf_nodes': '15', 'model__max_iter': '400', 'model__max_depth': '3', 'model__learning_rate': '0.05', 'model__l2_regularization': '1'}


In [ ]:
def to_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)

calibrator = LogisticRegression().fit(to_logit(models["logistic"].predict_proba(X_val)[:, 1]), y_val)
print("calibrator ready")

calibrator ready


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, brier_score_loss

metrics = {"data_source": "Synthetic data calibrated to TransUnion CIBIL published aggregates",
           "rows_used": int(len(df)), "sklearn_version": sklearn.__version__,
           "best_params": best_params, "models": {}}

for name, m in models.items():
    p = m.predict_proba(X_test)[:, 1]
    pred = (p >= 0.5).astype(int)
    tr_auc = roc_auc_score(y_train, m.predict_proba(X_train)[:, 1])
    te_auc = roc_auc_score(y_test, p)
    metrics["models"][name] = {
        "roc_auc_test": round(te_auc, 4), "roc_auc_train": round(tr_auc, 4),
        "train_test_gap": round(tr_auc - te_auc, 4),
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred), 4),
        "recall": round(recall_score(y_test, pred), 4),
        "brier": round(brier_score_loss(y_test, p), 4),
        "confusion_matrix": confusion_matrix(y_test, pred).tolist()}

p_lr = models["logistic"].predict_proba(X_test)[:, 1]
p_cal = calibrator.predict_proba(to_logit(p_lr))[:, 1]
metrics["logistic_calibrated_brier"] = round(brier_score_loss(y_test, p_cal), 4)

gain = metrics["models"]["boosting"]["roc_auc_test"] - metrics["models"]["logistic"]["roc_auc_test"]
metrics["boosting_auc_gain_over_logistic"] = round(gain, 4)
metrics["shipped_model"] = "logistic" if gain < 0.02 else "consider hybrid (boosting + logistic)"

print(pd.DataFrame(metrics["models"]).T.drop(columns="confusion_matrix"))
print("calibrated logistic brier:", metrics["logistic_calibrated_brier"])
print("boosting gain:", metrics["boosting_auc_gain_over_logistic"], "->", metrics["shipped_model"])

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

         roc_auc_test roc_auc_train train_test_gap accuracy precision  recall  \
logistic       0.8712        0.8747         0.0034   0.7875    0.7737  0.7456   
tree           0.8578        0.8642         0.0063   0.7805     0.735  0.8005   
boosting       0.8866        0.8937         0.0072   0.8018    0.7748  0.7885   

           brier  
logistic  0.1456  
tree      0.1506  
boosting   0.136  
calibrated logistic brier: 0.1455
boosting gain: 0.0154 -> logistic


In [ ]:
from sklearn.calibration import calibration_curve

r1, p1 = calibration_curve(y_test, p_lr, n_bins=10)
r2, p2 = calibration_curve(y_test, p_cal, n_bins=10)
print(pd.DataFrame({"raw_pred": p1.round(2), "raw_actual": r1.round(2),
                    "cal_pred": p2.round(2), "cal_actual": r2.round(2)}))

   raw_pred  raw_actual  cal_pred  cal_actual
0      0.04        0.02      0.04        0.02
1      0.15        0.14      0.15        0.14
2      0.25        0.27      0.25        0.27
3      0.35        0.38      0.35        0.39
4      0.45        0.48      0.45        0.47
5      0.55        0.57      0.55        0.57
6      0.65        0.67      0.65        0.67
7      0.75        0.78      0.75        0.78
8      0.85        0.85      0.85        0.84
9      0.94        0.91      0.94        0.91


In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(models["logistic"], X_test, y_test, scoring="roc_auc",
                            n_repeats=5, random_state=42, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)
print(imp.round(4))

cibil_score               0.0903
foir_post_loan            0.0450
is_new_to_credit          0.0329
missed_payments_12m       0.0313
monthly_income            0.0307
documents_complete        0.0219
foir_current              0.0099
credit_utilisation_pct    0.0067
employment_years          0.0020
recent_enquiries_6m       0.0019
existing_emi_monthly      0.0009
loan_to_monthly_income    0.0008
loan_amount_requested     0.0006
residence_years           0.0005
loan_tenure_months        0.0003
existing_loans_count      0.0001
credit_age_years         -0.0000
employment_type          -0.0000
dtype: float64


In [ ]:
from sklearn.tree import export_text

tree_pipe = models["tree"]
names = [n.split("__")[1] for n in tree_pipe.named_steps["prep"].get_feature_names_out()]
rules_text = export_text(tree_pipe.named_steps["model"], feature_names=names, max_depth=3)
print(rules_text)

|--- cibil_score <= 750.50
|   |--- cibil_score <= 694.50
|   |   |--- cibil_score <= 671.50
|   |   |   |--- cibil_score <= 655.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- cibil_score >  655.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |--- cibil_score >  671.50
|   |   |   |--- monthly_income <= 29750.00
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- monthly_income >  29750.00
|   |   |   |   |--- truncated branch of depth 2
|   |--- cibil_score >  694.50
|   |   |--- monthly_income <= 23750.00
|   |   |   |--- employment_years <= 2.25
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- employment_years >  2.25
|   |   |   |   |--- truncated branch of depth 2
|   |   |--- monthly_income >  23750.00
|   |   |   |--- documents_complete <= 0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- documents_complete >  0.50
|   |   |   |   |--- truncated branch of depth 2
|--- cibil_score >  750.50
|   

In [ ]:
d = df.copy()
d["age_band"] = pd.cut(d["age"], [0, 25, 35, 45, 55, 100], labels=["21-25", "26-35", "36-45", "46-55", "56+"])
d["income_band"] = pd.cut(d["monthly_income"], [0, 15000, 25000, 40000, 70000, 1e12],
                          labels=["<15k", "15-25k", "25-40k", "40-70k", "70k+"])
d["foir_bucket"] = pd.cut(d["foir_post_loan"], [0, 0.2, 0.35, 0.5, 100],
                          labels=["<20%", "20-35%", "35-50%", ">50%"])

def seg(col):
    g = d.groupby(col, observed=True)["approved"].agg(["mean", "size"])
    return {str(k): {"approval_rate": round(float(r["mean"]), 4), "n": int(r["size"])}
            for k, r in g.iterrows()}

reasons = (d.loc[d["approved"] == 0, "primary_rejection_reason"]
           .value_counts(normalize=True).round(4))

stats = {
    "data_source": "Synthetic data calibrated to TransUnion CIBIL published aggregates",
    "rows_used": int(len(d)),
    "overall_approval_rate": round(float(d["approved"].mean()), 4),
    "segments": {c: seg(c) for c in ["income_band", "cibil_tier", "employment_type",
                                     "foir_bucket", "city_tier", "age_band"]},
    "fairness": {c: seg(c) for c in ["gender", "city_tier", "age_band"]},
    "rejection_reasons": {str(k): float(v) for k, v in reasons.items()},
    "permutation_importance": {k: round(float(v), 4) for k, v in imp.items()},
    "tree_rules_text": rules_text,
}

with open("dashboard_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

for c, v in stats["fairness"].items():
    rates = [x["approval_rate"] for x in v.values()]
    print(c, "approval gap (max-min):", round(max(rates) - min(rates), 3))

gender approval gap (max-min): 0.003
city_tier approval gap (max-min): 0.157
age_band approval gap (max-min): 0.456


In [ ]:
from google.colab import files

shutil.copy("dashboard_stats.json", "pkg/models/dashboard_stats.json")
shutil.make_archive("finsight_phase2", "zip", "pkg")
files.download("finsight_phase2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

with open("metrics.json") as f:
    metrics = json.load(f)
with open("dashboard_stats.json") as f:
    stats = json.load(f)

print(json.dumps(metrics, indent=2))

{
  "data_source": "Synthetic data calibrated to TransUnion CIBIL published aggregates",
  "rows_used": 100000,
  "sklearn_version": "1.6.1",
  "best_params": {
    "logistic": {
      "model__C": "0.0745934328572655"
    },
    "tree": {
      "model__min_samples_leaf": "50",
      "model__max_depth": "5"
    },
    "boosting": {
      "model__min_samples_leaf": "50",
      "model__max_leaf_nodes": "15",
      "model__max_iter": "400",
      "model__max_depth": "3",
      "model__learning_rate": "0.05",
      "model__l2_regularization": "1"
    }
  },
  "models": {
    "logistic": {
      "roc_auc_test": 0.8712,
      "roc_auc_train": 0.8747,
      "train_test_gap": 0.0034,
      "accuracy": 0.7875,
      "precision": 0.7737,
      "recall": 0.7456,
      "brier": 0.1456,
      "confusion_matrix": [
        [
          6784,
          1471
        ],
        [
          1716,
          5030
        ]
      ]
    },
    "tree": {
      "roc_auc_test": 0.8578,
      "roc_auc_train": 0.8

In [ ]:
assert not set(FEATURES) & set(SENSITIVE), "sensitive column in features!"
print("policy test passed")

os.makedirs("pkg/models", exist_ok=True)
for name, m in models.items():
    joblib.dump(m, f"pkg/models/{name}.joblib")
joblib.dump(calibrator, "pkg/models/calibrator.joblib")
shutil.copy("metrics.json", "pkg/models/metrics.json")
shutil.copy("features.py", "pkg/features.py")
print(os.listdir("pkg/models"))

policy test passed
['boosting.joblib', 'tree.joblib', 'calibrator.joblib', 'metrics.json', 'logistic.joblib']


In [ ]:
m = joblib.load("pkg/models/logistic.joblib")
print("saved model test AUC:", round(roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]), 4))

saved model test AUC: 0.8712


In [ ]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(models["logistic"], X_test, y_test, scoring="roc_auc",
                            n_repeats=5, random_state=42, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values(ascending=False)
print(imp.round(4))

cibil_score               0.0903
foir_post_loan            0.0450
is_new_to_credit          0.0329
missed_payments_12m       0.0313
monthly_income            0.0307
documents_complete        0.0219
foir_current              0.0099
credit_utilisation_pct    0.0067
employment_years          0.0020
recent_enquiries_6m       0.0019
existing_emi_monthly      0.0009
loan_to_monthly_income    0.0008
loan_amount_requested     0.0006
residence_years           0.0005
loan_tenure_months        0.0003
existing_loans_count      0.0001
credit_age_years         -0.0000
employment_type          -0.0000
dtype: float64


In [ ]:
from sklearn.model_selection import cross_val_score

groups = {
    "credit":     ["cibil_score", "credit_age_years", "credit_utilisation_pct", "recent_enquiries_6m",
                   "missed_payments_12m", "existing_loans_count", "is_new_to_credit"],
    "income_loan": ["monthly_income", "loan_amount_requested", "loan_tenure_months", "existing_emi_monthly"],
    "ratios":     ["foir_current", "foir_post_loan", "loan_to_monthly_income"],
    "employment": ["employment_type", "employment_years", "residence_years"],
    "documents":  ["documents_complete"],
}

def build(cols):
    num = [c for c in cols if c in NUMERIC]
    cat = [c for c in cols if c in CATEGORICAL]
    parts = [("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num)]
    if cat:
        parts.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat))
    return Pipeline([("prep", ColumnTransformer(parts)), ("model", LogisticRegression(C=0.0746, max_iter=1000))])

def cv_auc(cols):
    return cross_val_score(build(cols), X_train[cols], y_train, cv=3, scoring="roc_auc", n_jobs=-1).mean()

full = cv_auc(FEATURES)
print("all 18 features:", round(full, 4))
for g, cols in groups.items():
    s = cv_auc([c for c in FEATURES if c not in cols])
    print(f"without {g:11s}: {s:.4f}   AUC lost: {full - s:+.4f}")

all 18 features: 0.8745
without credit     : 0.7826   AUC lost: +0.0919
without income_loan: 0.8674   AUC lost: +0.0071
without ratios     : 0.8721   AUC lost: +0.0024
without employment : 0.8738   AUC lost: +0.0007
without documents  : 0.8632   AUC lost: +0.0113


In [4]:
import json
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from google.colab import files

cols = ["credit_utilisation_pct", "missed_payments_12m", "recent_enquiries_6m",
        "credit_age_years", "monthly_income"]
d = df[df["is_new_to_credit"] == 0].dropna(subset=cols + ["cibil_score"])
tr, va = d[d["split"] == "train"], d[d["split"] == "val"]      # test set stays untouched

lin = LinearRegression().fit(tr[cols], tr["cibil_score"])
coef = dict(zip(cols, lin.coef_))
r2 = r2_score(va["cibil_score"], lin.predict(va[cols]))

print("rows used for fitting:", len(tr))
print("R2 on val:", round(r2, 3))
for k, v in coef.items():
    print(f"{k:26s} {v:+.4f}")

u = coef["credit_utilisation_pct"]
print(f"\nscore change per 1 point of card usage: {u:+.3f}")
print(f"card usage 85% -> 30%  = {u * -55:+.0f} score points (others held fixed)")
print(f"card usage 60% -> 30%  = {u * -30:+.0f} score points")

rule = {"per_utilisation_point": round(float(u), 4), "r2_val": round(float(r2), 3),
        "rows": int(len(tr)), "max_score": 900,
        "note": "Assumption estimated from synthetic data; not a real bureau rule."}
with open("score_rule.json", "w") as f:
    json.dump(rule, f, indent=2)
files.download("score_rule.json")

rows used for fitting: 7972
R2 on val: 0.575
credit_utilisation_pct     -1.2347
missed_payments_12m        -38.3342
recent_enquiries_6m        -12.6449
credit_age_years           +2.3743
monthly_income             +0.0003

score change per 1 point of card usage: -1.235
card usage 85% -> 30%  = +68 score points (others held fixed)
card usage 60% -> 30%  = +37 score points


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>